In [6]:
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [7]:
# Configuration and paths
mac = 20
pheno_list_type = None
# pheno_list_type = 'biochemistry'
# pheno_list_type = 'overall_phenotype'

# Load phenotype configuration
pheno_config_path = "/home/dnanexus/ukbgym/phenotype_config.yaml"
with open(pheno_config_path) as f:
    pheno_config = yaml.safe_load(f)

if pheno_list_type is not None:
    pheno_list = pheno_config[pheno_list_type]

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

# Load annotation configuration
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i8
"""plof""","""loftee_hc""","""#E31A1C""","""LOFTEE HC""",1
"""plof_consequences""","""consequence_frameshift_variant""","""#E31A1C""","""VEP Frameshift""",1
"""plof_consequences""","""consequence_stop_gained""","""#FB9A99""","""VEP Stop Gained""",1
"""plof_consequences""","""consequence_splice_donor_varia…","""#6A1B9A""","""VEP Splice Donor""",1
"""plof_consequences""","""consequence_splice_acceptor_va…","""#AB47BC""","""VEP Splice Acceptor""",1
…,…,…,…,…
"""vep_consequences""","""consequence_stop_lost""","""#FFB300""","""VEP Stop Lost""",1
"""vep_consequences""","""consequence_missense_variant""","""#1E90FF""","""VEP Missense""",1
"""vep_consequences""","""consequence_synonymous_variant""","""#C6DBEF""","""VEP Synonymous""",1


In [12]:
RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "/home/dnanexus/data_dir"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
    .select(['id', 'region', 'loftee_hc'])
    .collect(engine='streaming')
)

anno

Error: path "/home/dnanexus/data_dir/annotations_with_all.parquet" already
exists but -f/--overwrite was not set


id,region,loftee_hc
str,str,i8
"""chr5:58969474:T:C""","""ENSG00000113448""",0
"""chr4:113211828:C:A""","""ENSG00000145362""",0
"""chr2:157747637:T:C""","""ENSG00000115170""",0
"""chr1:214508142:C:G""","""ENSG00000152104""",0
"""chr15:45070714:C:A""","""ENSG00000140263""",0
…,…,…
"""chr17:64541107:C:T""","""ENSG00000108854""",0
"""chr4:168602088:G:A""","""ENSG00000129116""",0
"""chr1:169596134:C:A""","""ENSG00000174175""",0


In [13]:
melted_anno = (
    anno

    .unpivot(
        index=["id", "region"],
        on=['loftee_hc'],
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
)

melted_anno

id,region,annotation,annotation_score
str,str,str,f32
"""chr5:58969474:T:C""","""ENSG00000113448""","""loftee_hc""",0.0
"""chr4:113211828:C:A""","""ENSG00000145362""","""loftee_hc""",0.0
"""chr2:157747637:T:C""","""ENSG00000115170""","""loftee_hc""",0.0
"""chr1:214508142:C:G""","""ENSG00000152104""","""loftee_hc""",0.0
"""chr15:45070714:C:A""","""ENSG00000140263""","""loftee_hc""",0.0
…,…,…,…
"""chr17:64541107:C:T""","""ENSG00000108854""","""loftee_hc""",0.0
"""chr4:168602088:G:A""","""ENSG00000129116""","""loftee_hc""",0.0
"""chr1:169596134:C:A""","""ENSG00000174175""","""loftee_hc""",0.0


In [14]:
RAP_APPV_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "/home/dnanexus/data_dir"

# APPV_FILE = "loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet"
APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Chain the filter and the much faster semi join
appv = (
    appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('n_individuals') <= mac
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
)
# unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series()

Error: path "/home/dnanexus/data_dir/quant_pheno_INT_loftee_mac20_EURunrelated
_miss20per_appv_small.parquet" already exists but -f/--overwrite was not set


In [15]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [16]:
gene_trait_df['region'].n_unique(), gene_trait_df.shape[0]

(671, 2165)

In [17]:
# --- 1. Lazily prepare the filter keys ---
region_keys = gene_trait_df.lazy().select(pl.col('region').unique())
anno_ids_lazy = melted_anno.lazy().join(
    region_keys, on='region', how='semi'
).select(pl.col('id').unique())


# --- 2. Build the main lazy query plan step-by-step ---
# Start by filtering 'appv', then join everything together. Every operation returns another LazyFrame.
final_lazy_plan = (
    # A. Use a semi join for efficient filtering (more memory-safe than 'is_in')
    appv
    .join(anno_ids_lazy, on="id", how="semi")

    # B. Join the other two dataframes
    .join(
        melted_anno.lazy().drop([c for c in melted_anno.columns if 'is_nan' in c]),
        on="id", 
        how="inner"
    )
    .join(
        gene_trait_df.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )

    # C. Apply the ranking (window function) lazily
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['annotation_score', 'mean_pheno_value']
    )

    # D. Apply the group_by and aggregation lazily
    # .group_by(["region", "gene_name", "phenotype", "annotation"])
    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.corr("annotation_score_rank", "mean_pheno_value_rank", propagate_nans=True)
    )
)

# --- 3. Execute the ENTIRE plan at once ---
# This is the ONLY time data is computed. The streaming engine handles the whole complex query in memory-safe chunks.
print("Executing the full lazy plan with the streaming engine...")
correlation_df = final_lazy_plan.collect(engine='streaming')
correlation_df

Executing the full lazy plan with the streaming engine...


region,phenotype,annotation,n_variants,correlation
str,str,str,u64,f64
"""ENSG00000185989""","""lymphocyte_count_int""","""loftee_hc""",31223,-0.026809
"""ENSG00000166603""","""body_mass_index_bmi_impedance_…","""loftee_hc""",2361,0.052045
"""ENSG00000242515""","""direct_bilirubin_int""","""loftee_hc""",149334,0.017292
"""ENSG00000127334""","""diastolic_blood_pressure_autom…","""loftee_hc""",5401,-0.029036
"""ENSG00000140443""","""weight_impedance_int""","""loftee_hc""",75363,-0.014508
…,…,…,…,…
"""ENSG00000173064""","""creactive_protein_int""","""loftee_hc""",41733,0.02168
"""ENSG00000134030""","""leg_fat_percentage_left_int""","""loftee_hc""",70937,0.000383
"""ENSG00000092203""","""leg_fat_mass_left_int""","""loftee_hc""",8178,0.001976


In [19]:
corr_file = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations"
correlation_df.filter(pl.col('annotation')=='loftee_hc').write_parquet(f'/home/dnanexus/data_dir/{corr_file}.parquet')

In [20]:
!dx upload /home/dnanexus/data_dir/{corr_file}.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/{corr_file}.parquet

[===========================================================>] Uploaded 33,369 of 33,369 bytes (100%) /home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet
ID                                file-J6X60z0Jg0y5258xXx51zf7Y
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/REGENIE_results
Name                              regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.pa
                                  rquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Sun Mar  1 15:51:24 2026
Created by                        shubhankar
 via the job                      job-J6Vyf7QJg0y3xpbVZX365kF2
Last modified                 

In [21]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/{corr_file}.parquet -o /home/dnanexus/data_dir/

loftee_corrs = (
    pl.read_parquet(f'/home/dnanexus/data_dir/{corr_file}.parquet')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_dir']) 
)

loftee_corrs

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per_correlations.parquet" already exists but -f/--overwrite was not set


region,phenotype,loftee_corr,loftee_corr_dir
str,str,f64,f64
"""ENSG00000185989""","""lymphocyte_count_int""",-0.026809,-1.0
"""ENSG00000166603""","""body_mass_index_bmi_impedance_…",0.052045,1.0
"""ENSG00000242515""","""direct_bilirubin_int""",0.017292,1.0
"""ENSG00000127334""","""diastolic_blood_pressure_autom…",-0.029036,-1.0
"""ENSG00000140443""","""weight_impedance_int""",-0.014508,-1.0
…,…,…,…
"""ENSG00000173064""","""creactive_protein_int""",0.02168,1.0
"""ENSG00000134030""","""leg_fat_percentage_left_int""",0.000383,1.0
"""ENSG00000092203""","""leg_fat_mass_left_int""",0.001976,1.0
